# Create a data lake graph projection with PGQ syntax

This notebook demonstrates how to define a Neptune Analytics data lake projection using PGQ's `CREATE PROPERTY GRAPH` statement— a declarative way to describe your vertices, edges, and properties in one statement, instead of hand-writing the Athena projection SQL.

We'll:
1. Register the source data as an Athena table.
2. Describe the graph with a `CREATE PROPERTY GRAPH` statement.
3. Translate it to projection SQL and import it into Neptune Analytics.
4. Run a graph algorithm on the result.

See the final section for a side-by-side comparison with the equivalent
plain Athena SQL.

## Setup

Import the necessary libraries and set up logging.

In [ ]:
# Run this cell if kagglehub is not installed (e.g. on SageMaker)
import sys
!{sys.executable} -m pip install -q kagglehub dotenv

In [ ]:
# Check the Python version:
from sys import version_info
assert version_info >= (3, 11), "Python 3.11 or higher is required"

import logging
import os
import kagglehub
import boto3
import dotenv
from pathlib import Path

from nx_neptune import empty_s3_bucket
from nx_neptune.session_manager import SessionManager
from nx_neptune.instance_management import execute_athena_query
from nx_neptune.utils.utils import get_stdout_logger, validate_and_get_env

In [ ]:
# Configure logging to see detailed information about the instance creation process
logger = get_stdout_logger(__name__, [
    'nx_neptune.instance_management',
    'nx_neptune.session_manager',
    __name__
])

## Configuration

Check for environment variables necessary for the notebook.

In [ ]:
dotenv.load_dotenv()
# Check for optional environment variables
env_vars = validate_and_get_env([
    'NETWORKX_S3_IMPORT_BUCKET_PATH',
    'NETWORKX_S3_TABLES_DATABASE',
    'NETWORKX_S3_TABLES_TABLENAME',
    'NETWORKX_STAGING_BUCKET',
    # Required to execute opencypher query outside of VPC.
    'NETWORKX_PUBLIC_CONNECTIVITY',
    'NETWORKX_DELETION_PROTECTION',
])

# Get environment variables
s3_location_import = os.getenv('NETWORKX_S3_IMPORT_BUCKET_PATH')
s3_staging_bucket = os.getenv('NETWORKX_STAGING_BUCKET')
s3_tables_catalog = os.getenv('NETWORKX_S3_TABLES_CATALOG')
s3_tables_database = os.getenv('NETWORKX_S3_TABLES_DATABASE')
s3_tables_tablename = os.getenv('NETWORKX_S3_TABLES_TABLENAME')
session_name = "nx-athena-test-full"

## Data Setup

PaySim data is available from [kaggle](https://www.kaggle.com/code/kartik2112/fraud-detection-on-paysim-dataset/input?select=PS_20174392719_1491204439457_log.csv).

Data should be uploaded to an S3 bucket, and an athena table created for that bucket.

The PaySim dataset includes a simulated mobile money dataset, that involves transactions between client actors and banks. We can use this dataset to detect fraudulent activities in the simulated data.

In [ ]:
paysim_s3_bucket = s3_staging_bucket.replace('s3://', '').strip('/')
paysim_s3_bucket_path = 'datasets/paysim/'

# Download the latest version of paysim data
paysim_path = Path(kagglehub.dataset_download("ealaxi/paysim1"))
print("Path to paysim dataset files:", paysim_path)

# upload CSV to an S3 bucket if necessary
s3_client = boto3.client('s3')

for file_path in filter(Path.is_file, paysim_path.iterdir()):
  key = f"{paysim_s3_bucket_path}{file_path.name}"
  try:
      s3_client.head_object(Bucket=paysim_s3_bucket, Key=key)
      print(f"skip (exists): {key}")
  except ClientError:
      print(f"uploading: {key}")
      s3_client.upload_file(str(file_path), paysim_s3_bucket, key)


### Register the dataset to Athena

In [ ]:
# create CSV table from the uploaded data
s3_bucket_location = f"s3://{paysim_s3_bucket}/{paysim_s3_bucket_path}"

create_csv_table_stmt = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {s3_tables_tablename} (
    `step` int,
    `type` string,
    `amount` float,
    `nameOrig` string,
    `oldbalanceOrg` float,
    `newbalanceOrig` float,
    `nameDest` string,
    `oldbalanceDest` float,
    `newbalanceDest` float,
    `isFraud` int,
    `isFlaggedFraud` int
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe'
WITH SERDEPROPERTIES ('field.delim' = ',')
STORED AS INPUTFORMAT 'org.apache.hadoop.mapred.TextInputFormat'
OUTPUTFORMAT 'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat'
LOCATION '{s3_bucket_location}'
TBLPROPERTIES ('classification' = 'csv', 'skip.header.line.count'='1');
"""

tasks = await execute_athena_query(
    create_csv_table_stmt,
    f"s3://{paysim_s3_bucket}/athena-results",
    database=s3_tables_database,
    catalog=s3_tables_catalog)

if tasks[0].done():
    print ("Create table completed")

## Obtain a Neptune Analytics Instance

Provision a new Neptune Analytics instance on demand, or retrieve an existing neptune-graph.

In [ ]:
session = SessionManager.session(session_name)
graph_list = session.list_graphs()
print("The following graphs are available:")
for g in graph_list:
    print(g)

In [ ]:
session = SessionManager.session(session_name)
graph = await session.get_or_create_graph(config={"provisionedMemory": 32})
print(f"Retrieved graph: {graph}")

## Route 1: Translate only

The library offers two routes for creating a PGQ graph projection. 


This first route translates the `CREATE PROPERTY GRAPH` statement into Athena projection SQL without running an import — useful for validating your PGQ against the actual data lake before committing to a full graph creation.


You can inspect the generated queries, confirm the tables and columns resolve, and catch schema mismatches early.

In [ ]:
DDL = f"""
CREATE PROPERTY GRAPH financial
  VERTEX TABLES (
    {s3_tables_tablename} AS sender
      KEY (nameOrig) LABEL customer NO PROPERTIES,
    {s3_tables_tablename} AS recipient
      KEY (nameDest) LABEL customer NO PROPERTIES
  )
  EDGE TABLES (
    {s3_tables_tablename}
      SOURCE KEY (nameOrig) REFERENCES sender
      DESTINATION KEY (nameDest) REFERENCES recipient
      LABEL transfer PROPERTIES ALL COLUMNS
  )
"""

from nx_neptune import AthenaTableMetadata, property_graph_to_sql

metadata = AthenaTableMetadata(catalog=s3_tables_catalog, database=s3_tables_database)
queries = property_graph_to_sql(DDL, metadata)
for q in queries:
    print ("-------")
    print (q)

## Route 2: Translate And Import

In a production setting, you can streamline graph creation with a single call to `import_from_graph_schema()`, which performs the PGQ translation and then runs the import — no need to generate and hand off the SQL yourself.

In [ ]:
paysim_import_dir = f"{s3_location_import}/import_paysim/"
# Clear any staged CSV from a previous run.
empty_s3_bucket(paysim_import_dir)

DDL = f"""
CREATE PROPERTY GRAPH financial
  VERTEX TABLES (
    {s3_tables_tablename} AS sender
      KEY (nameOrig) LABEL customer NO PROPERTIES,
    {s3_tables_tablename} AS recipient
      KEY (nameDest) LABEL customer NO PROPERTIES
  )
  EDGE TABLES (
    {s3_tables_tablename}
      SOURCE KEY (nameOrig) REFERENCES sender
      DESTINATION KEY (nameDest) REFERENCES recipient
      LABEL transfer PROPERTIES ALL COLUMNS
  )
"""

await session.import_from_graph_schema(
    graph,
    paysim_import_dir,
    DDL,
    catalog=s3_tables_catalog,
    database=s3_tables_database
)


## Run a graph algorithm

The projection is now a live Neptune Analytics graph. As a quick demonstration that it imported correctly, we run Louvain community detection over it and write each vertex's community back as a community property.

In [ ]:
# Sanity check: the graph has vertices and edges from the PGQ projection.
print(graph.execute_query("MATCH (n) RETURN count(n) AS nodes"))
print(graph.execute_query("MATCH ()-[r]->() RETURN count(r) AS edges"))

# Run Louvain and write each vertex's community into the "community" property.
louvain = graph.execute_query('''
    CALL neptune.algo.louvain.mutate({iterationTolerance:1e-07, writeProperty:"community"}) 
    YIELD success AS success 
    RETURN success''')
print(f"Louvain: {louvain}")


In [ ]:
# destroy the session graphs
session.destroy_all_graphs()

## Conclusion

This notebook demonstrated the complete lifecycle of running analytics from a datalake projection into Neptune Analytics instance:

1. **Creation**: We created a new Neptune Analytics instance on demand
2. **Import**: We imported a projection of the datalake
3. **Usage**: We ran graph algorithms (Louvain) on the instance and mutated the data
4. **Deletion**: We exported the updated data back into the datalake into an iceberg table

The session manager (`SessionManager`) provides an easy mechanism to execute general datalake functionality.